In [1]:
#! /usr/bin/env python3
import numpy as np
import sys
import h5py

sys.path.append("build/python")
sys.path.append(".")

from canoe import def_species, load_configure, index_map
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")

nx2 = 1 
pin.set_boolean("job", "verbose", False)
pin.set_string("mesh", "nx2", f"{nx2}")

pindex = index_map.get_instance()
iNH3 = pindex.get_vapor_id("NH3")

mesh = Mesh(pin) 
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)  ## fetch the first block of the mesh
P0 = pin.get_real("mesh", "ReferencePressure")

## construct atmosphere
## deep layer params
qNH3=354.96
qH2O=1790
T1bar=177.91

## use the first ap column of the block
Jindex=0 

## RH limit 
RHmax=0.71
# adlnNH3dlnP=-0.06
# pmin=0
# pmax=4.88E5


mb.construct_atmosphere(pin, qNH3, T1bar, RHmax, Jindex, "pseudo", qH2O, 500)


## get all radiance，4 outdirections
# rad = mb.get_rad()
# nb = rad.get_num_bands()
# rad.cal_radiance(mb, mb.k_st, mb.j_st + Jindex)
# tb = np.array([0.0] * 4 * nb)

# for ib in range(nb):
#     toa = rad.get_band(ib).get_toa()[0]
#     tb[ib * 4 : ib * 4 + 4] = toa

# calc radiance
rad = mb.get_rad()
rad.cal_radiance(mb, mb.k_st, mb.j_st + Jindex)

# ## output into nc
print(pin.set_string("job", "problem_id","juno_mwr_moistdiab_rhmax"))
out = Outputs(mesh, pin)
out.make_outputs(mesh, pin)


Log, "2025-09-21 17:40:59",        canoe, 1., "Installing monitor canoe"
Log, "2025-09-21 17:40:59",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-21 17:40:59",         snap, 3., "Installing monitor snap"
Log, "2025-09-21 17:40:59",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-21 17:40:59",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-21 17:40:59",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-21 17:40:59",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-21 17:40:59",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-21 17:40:59",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-21 17:40:59", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-21 17:40:59", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-21 17:40:59",         harp, 9., "Installing monitor harp"
Log, "2025-09-21 17:40:59",         harp, 9.1., "Initialize Radiation"
Log, "2025-09-2